# 4th Place Solution - AMP Parkinson's Disease 
This is my 4th place solution to Kaggle's AMP®-Parkinson's Disease Progression Prediction competition. The discussion describing this solution is [here][1]

[1]: https://www.kaggle.com/competitions/amp-parkinsons-disease-progression-prediction/discussion/411398

# Load Data

In [ ]:
import pandas as pd
import numpy as np, gc

# PATIENT VISIT DATA
df1 = pd.read_csv('/kaggle/input/amp-parkinsons-disease-progression-prediction/train_clinical_data.csv')
print('Patient Visit Data shape', df1.shape )
df1.head()

In [ ]:
# PATIENT PROTEIN DATA
df2 = pd.read_csv('/kaggle/input/amp-parkinsons-disease-progression-prediction/train_proteins.csv')
print('Patient Protein Data shape', df2.shape )
df2.head()

In [ ]:
# TARGETS
TARS = ['updrs_1', 'updrs_2', 'updrs_3', 'updrs_4']

In [ ]:
#PATIENTS
PATS = df1.patient_id.unique()
print('There are',len( PATS ),'unique patients')

# Feature Engineer
We will add "booleans" indicating whether a patient visited doctor on visit month `[0,6,12,18,24,36,48,60,72,84]`. These "booleans" are descibed in discussion [here][1]

[1]: https://www.kaggle.com/competitions/amp-parkinsons-disease-progression-prediction/discussion/411398

In [ ]:
# THESE ARE THE VISIT MONTHS IN KAGGLE'S API
USE = [0,6,12,18,24,36,48,60,72,84]

In [ ]:
rows = []

print('Processing',len(PATS),'patients...')
for jj,p in enumerate(PATS):
    print(jj,', ',end='')
    tmp1 = df1.loc[df1.patient_id==p]
    m = df2.loc[df2.patient_id==p,'visit_month'].min()
    vm = np.zeros(len(USE))
    for i,t in enumerate(USE):
        if t in tmp1.visit_month.values:
            vm[i] = 1
    for index,row in tmp1.iterrows():
        dd = {}
        dd['patient_id'] = p
        dd['visit_month'] = row.visit_month
        for t in TARS:
            dd[t] = row[t]
        dd['prot'] = m
        for i,t in enumerate(USE):
            dd[f'v{t}'] = vm[i]
        rows.append(dd)

In [ ]:
# REMOVE DATA BEFORE FIRST BLOOD WORK (because it is not scored by Kaggle)
df2 = pd.DataFrame(rows)
df2 = df2.loc[df2.visit_month >= df2.prot].reset_index(drop=True)
print( df2.shape )
df2.head()

In [ ]:
# CONVERT FUTURE VISITS AS UNKNOWN
for k in range(len(df2)):
    v = df2.loc[k,'visit_month']
    for j in USE:
        if j>v:
            df2.loc[k,f'v{j}'] = -1

# Create Train Data
We need to replace each row in original data with **4 new rows**. This is explained in discussion [here][1]

[1]: https://www.kaggle.com/competitions/amp-parkinsons-disease-progression-prediction/discussion/411398

In [ ]:
COLS = df2.columns

rows = []
for index,row in df2.iterrows():
    
    dd = {}
    for c in COLS:
        dd[c] = row[c]
    rows.append(dd)
    
    v = row.visit_month
    
    if v==6:
        if row.v0==1:
            dd = {}
            for c in COLS:
                dd[c] = row[c]
            dd['v6'] = -1
            rows.append(dd)
            
    if v==12:
        if row.v6==1:
            dd = {}
            for c in COLS:
                dd[c] = row[c]
            dd['v12'] = -1
            rows.append(dd)
        if row.v0==1:
            dd = {}
            for c in COLS:
                dd[c] = row[c]
            dd['v6'] = -1
            dd['v12'] = -1
            rows.append(dd)
            
    if v==18:
        if row.v12==1:
            dd = {}
            for c in COLS:
                dd[c] = row[c]
            dd['v18'] = -1
            rows.append(dd)
        if row.v6==1:
            dd = {}
            for c in COLS:
                dd[c] = row[c]
            dd['v12'] = -1
            dd['v18'] = -1
            rows.append(dd)
            
    if v==24:
        if row.v18==1:
            dd = {}
            for c in COLS:
                dd[c] = row[c]
            dd['v24'] = -1
            rows.append(dd)
        if row.v12==1:
            dd = {}
            for c in COLS:
                dd[c] = row[c]
            dd['v18'] = -1
            dd['v24'] = -1
            rows.append(dd)
        if row.v0==1:
            dd = {}
            for c in COLS:
                dd[c] = row[c]
            dd['v6'] = -1
            dd['v12'] = -1
            dd['v18'] = -1
            dd['v24'] = -1
            rows.append(dd)
            
    # IS THIS SCORED??
    if v==30:
        if row.v18==1:
            dd = {}
            for c in COLS:
                dd[c] = row[c]
            dd['v24'] = -1
            rows.append(dd)
        if row.v6==1:
            dd = {}
            for c in COLS:
                dd[c] = row[c]
            dd['v12'] = -1
            dd['v18'] = -1
            dd['v24'] = -1
            rows.append(dd)
            
    if v==36:
        if row.v24==1:
            dd = {}
            for c in COLS:
                dd[c] = row[c]
            dd['v36'] = -1
            rows.append(dd)
        if row.v12==1:
            dd = {}
            for c in COLS:
                dd[c] = row[c]
            dd['v18'] = -1
            dd['v24'] = -1
            dd['v36'] = -1
            rows.append(dd)
            
    # IS THIS SCORED ??
    if v==42:
        if row.v18==1:
            dd = {}
            for c in COLS:
                dd[c] = row[c]
            dd['v24'] = -1
            dd['v36'] = -1
            rows.append(dd)
            
    if v==48:
        if row.v36==1:
            dd = {}
            for c in COLS:
                dd[c] = row[c]
            dd['v48'] = -1
            rows.append(dd)
        if row.v24==1:
            dd = {}
            for c in COLS:
                dd[c] = row[c]
            dd['v36'] = -1
            dd['v48'] = -1
            rows.append(dd)
                        
    if v==60:
        if row.v48==1:
            dd = {}
            for c in COLS:
                dd[c] = row[c]
            dd['v60'] = -1
            rows.append(dd)
        if row.v36==1:
            dd = {}
            for c in COLS:
                dd[c] = row[c]
            dd['v48'] = -1
            dd['v60'] = -1
            rows.append(dd)
            
    if v==72:
        if row.v60==1:
            dd = {}
            for c in COLS:
                dd[c] = row[c]
            dd['v72'] = -1
            rows.append(dd)
        if row.v48==1:
            dd = {}
            for c in COLS:
                dd[c] = row[c]
            dd['v60'] = -1
            dd['v72'] = -1
            rows.append(dd)
            
    if v==84:
        if row.v72==1:
            dd = {}
            for c in COLS:
                dd[c] = row[c]
            dd['v84'] = -1
            rows.append(dd)
        if row.v60==1:
            dd = {}
            for c in COLS:
                dd[c] = row[c]
            dd['v72'] = -1
            dd['v84'] = -1
            rows.append(dd)
            
    if v==96:
        if row.v72==1:
            dd = {}
            for c in COLS:
                dd[c] = row[c]
            dd['v84'] = -1
            rows.append(dd)

In [ ]:
# UN-STANDARIZED FEATURES
df2 = pd.DataFrame(rows)
print( df2.shape )
df2.head(10)

# Standardize Features
We use 11 features. Both SVR and MLP performs better if features are standardized

In [ ]:
FEATURES = ['visit_month'] + [f'v{x}' for x in USE] 
print( len(FEATURES) )
print( FEATURES )

In [ ]:
# STANDARIZE FEATURES
mms = []
sss = []

FEATURES2 = []
for k,f in enumerate(FEATURES):
    
    m = df2[f].mean()
    mms.append(m)
    s = df2[f].std()
    sss.append(s)
    
    n = f'f{k}'
    if s==0:
        df2[n] = (df2[f]-m)
    else:
        df2[n] = (df2[f]-m)/s
    FEATURES2.append(n)
print( len(FEATURES2) )
print( FEATURES2 )

In [ ]:
# OUR TRAIN DATA
data = df2.astype('float32')
print( data.shape )
data.head()

# Train TensorFlow MLP
We train MLP with 10 hidden layer of size 24 each

In [ ]:
import tensorflow as tf
import tensorflow.keras.backend as K
tf.__version__

In [ ]:
# OUR MODEL
HIDDEN_LAYERS = 10
HIDDEN_SIZE = 24
ACT = 'relu'

def build_model():
    inp = tf.keras.Input(shape=(len(FEATURES2,)))
    
    x = tf.keras.layers.Dense(HIDDEN_SIZE,activation=ACT)(inp)
    for k in range(HIDDEN_LAYERS-1):
        x = tf.keras.layers.Dense(HIDDEN_SIZE)(x)
        x = tf.keras.layers.Activation(ACT)(x)
    x = tf.keras.layers.Dense(1,activation='linear')(x)
    
    model = tf.keras.Model(inputs=[inp], outputs=[x])
    opt = tf.keras.optimizers.Adam(learning_rate=0.001)
    loss = tf.keras.losses.MeanAbsoluteError()
    model.compile(loss=loss, optimizer = opt)
    
    return model

In [ ]:
# TRAIN OUR MODEL
models = {}

for TAR in TARS:
    print('#'*25)
    print('###',TAR)
    print('#'*25)

    models[TAR] = []

    for fold in range(5):
        train_idx = np.arange(len(data))

        print('=> Fold',fold+1)
        print('   Train size',len(train_idx)) 
        #print()

        RMV = data.loc[data[TAR].isna()].index
        train_idx = np.setdiff1d(train_idx,RMV)

        X_train = data.loc[train_idx,FEATURES2]
        y_train = data.loc[train_idx,TAR]

        clf = build_model()
        clf.fit(X_train,y_train,
                #validation_data = (X_valid,y_valid),
                verbose=2, epochs=12, batch_size=32)
        opt = tf.keras.optimizers.Adam(learning_rate=0.0001)
        loss = tf.keras.losses.MeanAbsoluteError()
        clf.compile(loss=loss, optimizer = opt)
        clf.fit(X_train,y_train,
                #validation_data = (X_valid,y_valid),
                verbose=2, epochs=12, batch_size=32)

        clf.save_weights(f'MLP_{TAR}_f{fold}.h5')
        
        del clf
        K.clear_session()
        gc.collect()

# Infer Test with Kaggle API

In [ ]:
import sys
sys.path.append('/kaggle/input/amp-pd')

import amp_pd_peptide_310
env = amp_pd_peptide_310.make_env()
iter_test = env.iter_test()

In [ ]:
# SNAP PREDICTIONS TO INTEGERS TO BOOST SMAPE
# WE COULD TRY ROUNDING AT 0.45 OR 0.40 AND MAYBE THAT IS BETTER
import math
def rnd(x):
    a,b = math.modf(x)
    if a>=0.5: return max(b+1,0)
    else: return max(b,0)

In [ ]:
p_ct = {}

# The API will deliver four dataframes in this specific order:
for (test, test_peptides, test_proteins, sample_submission) in iter_test:
    
    # CLEAR GPU MEMORY. LOAD TF MODELS
    del models
    gc.collect()
    K.clear_session()
    models = {}
    for t in TARS:
        models[t] = []
        for fold in range(5):
            clf = build_model()
            clf.load_weights(f'MLP_{t}_f{fold}.h5')
            models[t].append(clf)
        
    # PATIENTS AND CURRENT VISIT MONTH
    PATS = test.patient_id.unique()
    VM = test.visit_month.values[0]
    print(VM,', ',end='')
    
    # RECORD VISIT DATES SEEN
    for p in PATS:
        if not p in p_ct: 
            p_ct[p] = {}
            for u in USE:
                if u <= VM:
                    p_ct[p][u] = 0
                else:
                    p_ct[p][u] = -1
        p_ct[p][VM] = 1
    # RECORD VISIT DATES UNSEEN
    for k in list(p_ct.keys()):
        if p_ct[k][VM] == -1:
            p_ct[k][VM] = 0

    #ITERATE PATIENTS AND MAKE PREDICTIONS
    result = []
    for p in PATS:
        
        # ITERATE 4 TARGETS
        for k in range(1,5):
            # ITERATE 4 FUTURE PREDICTIONS
            for j in [0,6,12,24]:
                dd = {}
                n = f'{p}_{VM}_updrs_{k}_plus_{j}_months'
                dd['prediction_id'] = n

                #########
                # PREDICT TARGET
                x1 = ((VM + j)-mms[0])/sss[0]
                x2 = np.zeros((1,len(USE)+1))
                x2[0,0] = x1
                for i,kk in enumerate(USE):
                    x2[0,i+1] = p_ct[p][kk]
                    s = sss[i+1]
                    if s==0:
                        x2[0,i+1] = x2[0,i+1]-mms[i+1]
                    else:
                        x2[0,i+1] = (x2[0,i+1]-mms[i+1])/sss[i+1]
                        
                ###########
                # INFER 5 FOLDS
                y = models[TARS[k-1]][0].predict(x2,verbose=0)[0]
                for kk in range(1,5):
                    y += models[TARS[k-1]][kk].predict(x2,verbose=0)[0]
                y = rnd( y/5.0 )
                    
                dd['rating'] = y
                #########

                result.append(dd)
    result = pd.DataFrame(result)

    env.predict(result)

In [ ]:
sub = pd.read_csv('submission.csv')
print( sub.shape )
sub.head()